# 05_MobileNetV2_YOLO_Seg

This notebook follows the same MobileNetV2 encoder strategy from the original segmentation notebook. Only the decoder/head is changed for fair benchmarking.


In [1]:
!pip install -q kaggle

import os
import shutil

# Your uploaded Kaggle API file
src = "/content/kaggle (2).json"

# Kaggle expects the file here with this exact name
kaggle_dir = "/root/.kaggle"
os.makedirs(kaggle_dir, exist_ok=True)

dst = os.path.join(kaggle_dir, "kaggle.json")
shutil.copy(src, dst)

# Required permission
os.chmod(dst, 0o600)

print("Kaggle API key installed successfully.")

Kaggle API key installed successfully.


In [2]:
!mkdir -p /content/brisc
!kaggle datasets download -d briscdataset/brisc2025 -p /content/brisc --unzip

Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:16<00:00, 16.1MB/s]



In [3]:
# =========================
# INSTALL / IMPORTS
# =========================
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_v2_preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import AUC
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow:", tf.__version__)


from tensorflow.keras import mixed_precision

# Set mixed precision policy
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

print("Mixed precision enabled:", mixed_precision.global_policy())


TensorFlow: 2.20.0
Mixed precision enabled: <DTypePolicy "mixed_float16">


In [4]:
from pathlib import Path

DATA_ROOT = Path("/content/brisc")

for p in DATA_ROOT.iterdir():
    print(p)

/content/brisc/brisc2025


In [5]:
# =========================
# REPRODUCIBILITY
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow determinism enabled.")
except Exception as e:
    print("Could not enable determinism:", e)

TensorFlow determinism enabled.


In [6]:
# =========================
# CONFIG
# =========================
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 1
INPUT_SHAPE = (224, 224, 3)

EMBED_DIM = 96
NUM_HEADS = 4
TRANSFORMER_DEPTH = 1
MLP_DIM = 192

PRIMARY_CAPS = 12
PRIMARY_CAPS_DIM = 8
CLASS_CAPS_DIM = 12
ROUTING_ITERS = 2
NUM_CLASSES = 4

ALPHA_CE = 0.9
BETA_MARGIN = 0.1
DROPOUT = 0.30
LABEL_SMOOTHING = 0.08
REG = tf.keras.regularizers.l2(1e-4)

VAL_SIZE = 0.15

EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 20

In [7]:
# =========================
# MODEL COMPONENTS
# =========================

def squash(vectors, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = s_squared_norm / (1.0 + s_squared_norm)
    return scale * vectors / tf.sqrt(s_squared_norm + epsilon)


class PatchTokenization(layers.Layer):
    def __init__(self, patch_size=2, embed_dim=EMBED_DIM, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = layers.Dense(embed_dim)

    def build(self, input_shape):
        _, h, w, c = input_shape

        self.num_patches_h = h // self.patch_size
        self.num_patches_w = w // self.patch_size
        self.num_tokens = self.num_patches_h * self.num_patches_w

        self.pos_embed = self.add_weight(
            name="pos_embed",
            shape=(1, self.num_tokens, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

        super().build(input_shape)

    def call(self, x):
        patches = tf.image.extract_patches(
            images=x,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        batch_size = tf.shape(patches)[0]
        patch_dim = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [batch_size, self.num_tokens, patch_dim]
        )

        tokens = self.proj(patches)

        return tokens + self.pos_embed


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=DROPOUT, **kwargs):
        super().__init__(**kwargs)

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.drop1 = layers.Dropout(dropout)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.mlp = tf.keras.Sequential([
            layers.Dense(
                mlp_dim,
                activation="gelu",
                kernel_regularizer=REG
            ),
            layers.Dropout(dropout),
            layers.Dense(
                embed_dim,
                kernel_regularizer=REG
            ),
            layers.Dropout(dropout)
        ])

    def call(self, x, training=False, return_attention=False):
        x_norm = self.norm1(x)

        attn_out, attn_scores = self.attn(
            x_norm,
            x_norm,
            return_attention_scores=True,
            training=training
        )

        x = x + self.drop1(attn_out, training=training)
        x = x + self.mlp(self.norm2(x), training=training)

        if return_attention:
            return x, attn_scores

        return x


class PrimaryCapsule(layers.Layer):
    def __init__(self, num_capsules=16, capsule_dim=8, **kwargs):
        super().__init__(**kwargs)

        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim

        self.proj = layers.Dense(
            num_capsules * capsule_dim,
            kernel_regularizer=REG
        )

    def call(self, tokens):
        x = self.proj(tokens)

        batch_size = tf.shape(x)[0]
        num_tokens = tf.shape(x)[1]

        x = tf.reshape(
            x,
            [batch_size, num_tokens, self.num_capsules, self.capsule_dim]
        )

        return squash(x)


class RelevanceAwareClassCapsule(layers.Layer):
    def __init__(self, num_classes, class_caps_dim=CLASS_CAPS_DIM, routing_iters=ROUTING_ITERS, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.class_caps_dim = class_caps_dim
        self.routing_iters = routing_iters

    def build(self, input_shape):
        _, num_tokens, num_primary, primary_dim = input_shape[0]
        self.W = self.add_weight(
            shape=(1, num_tokens, num_primary, self.num_classes, self.class_caps_dim, primary_dim),
            initializer="glorot_uniform",
            trainable=True,
            name="capsule_transform"
        )
        super().build(input_shape)

    def call(self, inputs):
        # Force float32 to prevent mixed precision errors
        u = tf.cast(inputs[0], tf.float32)  # [B, num_tokens, num_primary, caps_dim]
        relevance = tf.cast(inputs[1], tf.float32)  # [B, num_tokens]

        batch_size = tf.shape(u)[0]
        num_tokens = tf.shape(u)[1]
        num_primary = tf.shape(u)[2]
        caps_dim = tf.shape(u)[3]
        num_classes = self.num_classes

        # Initialize routing logits
        b = tf.zeros([batch_size, num_classes, num_tokens], dtype=tf.float32)

        for i in range(self.routing_iters):
            c = tf.nn.softmax(b, axis=1)  # [B, num_classes, num_tokens]
            c = c * relevance[:, tf.newaxis, :]  # broadcast relevance
            c = c / (tf.reduce_sum(c, axis=-1, keepdims=True) + 1e-8)

            # Compute weighted sum of primary capsules: u_hat = sum(c * u)
            # einsum: c [B, C, N], u [B, N, P, D] -> s [B, C, P, D]
            s = tf.einsum('bcn,bnid->bcd', c, u)  # sum over tokens

            # Apply squash function
            v = squash(s)

            # Update logits b if not last iteration
            if i < self.routing_iters - 1:
                b += tf.einsum('bcd,bnid->bcn', v, u)

        return v  # shape [B, num_classes, num_primary, class_caps_dim]


class MobileNetV2Hybrid(Model):
    def __init__(
        self,
        num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        transformer_depth=TRANSFORMER_DEPTH,
        mlp_dim=MLP_DIM,
        primary_caps=PRIMARY_CAPS,
        primary_caps_dim=PRIMARY_CAPS_DIM,
        class_caps_dim=CLASS_CAPS_DIM,
        routing_iters=ROUTING_ITERS,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_classes = num_classes

        self.backbone = MobileNetV2(
            weights="imagenet",
            include_top=False,
            input_shape=(224, 224, 3)
        )

        self.backbone.trainable = False

        self.proj_conv = layers.Conv2D(
            96,
            1,
            padding="same",
            use_bias=False,
            kernel_regularizer=REG
        )

        self.proj_bn = layers.BatchNormalization()
        self.proj_act = layers.Activation("swish")
        self.proj_dropout = layers.Dropout(0.25)

        self.tokenizer = PatchTokenization(
            patch_size=2,
            embed_dim=embed_dim
        )

        self.transformer_blocks = [
            TransformerBlock(
                embed_dim,
                num_heads,
                mlp_dim,
                dropout=DROPOUT,
                name=f"transformer_block_{i}"
            )
            for i in range(transformer_depth)
        ]

        self.final_norm = layers.LayerNormalization(epsilon=1e-6)

        self.token_relevance_head = layers.Dense(
            1,
            activation="sigmoid",
            kernel_regularizer=REG
        )

        self.primary_caps = PrimaryCapsule(
            primary_caps,
            primary_caps_dim
        )

        self.class_caps = RelevanceAwareClassCapsule(
            num_classes,
            class_caps_dim,
            routing_iters
        )

        self.last_conv_feature_map = None
        self.last_attention_scores = None
        self.last_routing_coeffs = None

    def call(self, inputs, training=False, return_extras=False):
        feature_map = self.backbone(inputs, training=training)

        feature_map = self.proj_conv(feature_map)
        feature_map = self.proj_bn(feature_map, training=training)
        feature_map = self.proj_act(feature_map)
        feature_map = self.proj_dropout(feature_map, training=training)

        self.last_conv_feature_map = feature_map

        tokens = self.tokenizer(feature_map)

        attention_scores_list = []

        for i, blk in enumerate(self.transformer_blocks):
            if i == len(self.transformer_blocks) - 1:
                tokens, attn = blk(
                    tokens,
                    training=training,
                    return_attention=True
                )
                attention_scores_list.append(attn)
            else:
                tokens = blk(tokens, training=training)

        tokens = self.final_norm(tokens)

        token_relevance = tf.squeeze(
            self.token_relevance_head(tokens),
            axis=-1
        )

        if attention_scores_list:
            last_attn = attention_scores_list[-1]

            # last_attn shape:
            # [B, num_heads, num_tokens, num_tokens]
            mean_attn = tf.reduce_mean(last_attn, axis=1)

            # Token-level attention importance
            attn_importance = tf.reduce_mean(mean_attn, axis=1)

            fused_relevance = (
                0.5 * attn_importance +
                0.5 * token_relevance
            )
        else:
            fused_relevance = token_relevance

        fused_relevance = fused_relevance / (
            tf.reduce_sum(fused_relevance, axis=-1, keepdims=True) + 1e-8
        )

        primary_caps = self.primary_caps(tokens)

        class_caps, routing_coeffs = self.class_caps(
            [primary_caps, fused_relevance]
        )

        caps_lengths = tf.norm(class_caps, axis=-1)

        probs = tf.nn.softmax(caps_lengths, axis=-1)

        self.last_attention_scores = (
            attention_scores_list[-1]
            if attention_scores_list
            else None
        )

        self.last_routing_coeffs = routing_coeffs

        if return_extras:
            return {
                "logits": probs,
                "caps_lengths": caps_lengths,
                "class_capsules": class_caps,
                "feature_map": feature_map,
                "tokens": tokens,
                "relevance": fused_relevance,
                "attention_scores": self.last_attention_scores,
                "routing_coeffs": routing_coeffs
            }

        return probs

In [8]:
# =========================
# CUSTOM TRAINER
# =========================
# This version supports class_weight through sample_weight.
class HybridTrainer(Model):
    def __init__(self, backbone, alpha_ce=ALPHA_CE, beta_margin=BETA_MARGIN, **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone
        self.alpha_ce = alpha_ce
        self.beta_margin = beta_margin

        self.ce_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.ce_tracker = tf.keras.metrics.Mean(name="ce_loss")
        self.margin_tracker = tf.keras.metrics.Mean(name="margin_loss")
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        self.auc_metric = AUC(name="auc", curve="ROC", multi_label=True, num_labels=NUM_CLASSES)

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.ce_tracker,
            self.margin_tracker,
            self.acc_metric,
            self.auc_metric
        ]

    def capsule_margin_loss(self, y_true, y_pred, sample_weight=None):
        present_error = tf.square(tf.maximum(0.0, 0.9 - y_pred))
        absent_error = tf.square(tf.maximum(0.0, y_pred - 0.1))
        loss = y_true * present_error + 0.5 * (1.0 - y_true) * absent_error
        loss_per_sample = tf.reduce_sum(loss, axis=1)

        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, loss_per_sample.dtype)
            loss_per_sample = loss_per_sample * sample_weight
            return tf.reduce_sum(loss_per_sample) / (tf.reduce_sum(sample_weight) + 1e-8)

        return tf.reduce_mean(loss_per_sample)

    def train_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)

        with tf.GradientTape() as tape:
            y_pred = self.backbone(x, training=True)
            ce_loss = self.ce_fn(y, y_pred, sample_weight=sample_weight)
            margin_loss = self.capsule_margin_loss(y, y_pred, sample_weight=sample_weight)
            total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
            total_loss += tf.add_n(self.backbone.losses) if self.backbone.losses else 0.0

        grads = tape.gradient(total_loss, self.backbone.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.backbone.trainable_variables))

        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred, sample_weight=sample_weight)
        self.auc_metric.update_state(y, y_pred, sample_weight=sample_weight)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y, sample_weight = tf.keras.utils.unpack_x_y_sample_weight(data)

        y_pred = self.backbone(x, training=False)
        ce_loss = self.ce_fn(y, y_pred, sample_weight=sample_weight)
        margin_loss = self.capsule_margin_loss(y, y_pred, sample_weight=sample_weight)
        total_loss = self.alpha_ce * ce_loss + self.beta_margin * margin_loss
        total_loss += tf.add_n(self.backbone.losses) if self.backbone.losses else 0.0

        self.loss_tracker.update_state(total_loss)
        self.ce_tracker.update_state(ce_loss)
        self.margin_tracker.update_state(margin_loss)
        self.acc_metric.update_state(y, y_pred, sample_weight=sample_weight)
        self.auc_metric.update_state(y, y_pred, sample_weight=sample_weight)

        return {m.name: m.result() for m in self.metrics}

    def call(self, x, training=False):
        return self.backbone(x, training=training)


def build_hybrid_training_model():
    backbone = MobileNetV2Hybrid(
        num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        transformer_depth=TRANSFORMER_DEPTH,
        mlp_dim=MLP_DIM,
        primary_caps=PRIMARY_CAPS,
        primary_caps_dim=PRIMARY_CAPS_DIM,
        class_caps_dim=CLASS_CAPS_DIM,
        routing_iters=ROUTING_ITERS,
        name="MobileNetV2_Hybrid"
    )

    trainer = HybridTrainer(
        backbone,
        alpha_ce=ALPHA_CE,
        beta_margin=BETA_MARGIN,
        name="Hybrid_Trainer"
    )

    return trainer, backbone

## Task

Implement YOLO-style segmentation head with prototype mask generation/FPN style features.

